In [1]:
import sys, numpy, sklearn, torch
print(sys.version.split()[0], numpy.__version__, sklearn.__version__, torch.__version__)
print("CUDA:", torch.cuda.is_available(), "| MPS:", torch.backends.mps.is_available())

3.11.17 2.4.6 1.9.1 2.14.1+cpu
CUDA: False | MPS: False


In [7]:
import time
import numpy as np, torch

device = ("Cuda" if torch.cuda.is_available()
          else "mps" if torch.backends.mps.is_available() else "cpu")
print("PyTorch device:", device)

rng = np.random.default_rng(412)
x = rng.random(1_000_000); w = rng.random(1_000_000)
t0 = time.perf_counter()
s = sum(a * b for a, b in zip(x, w))
t1 = time.perf_counter()
v = x @ w
t2 = time.perf_counter()
print(f"loop {t1-t0:.3f}s | vectorised {t2-t1:.5f}s | "
      f"speed-up {(t1-t0)/(t2-t1):.0f}x | same: {np.isclose(s, v)}")

PyTorch device: cpu
loop 0.371s | vectorised 0.00091s | speed-up 409x | same: True


In [11]:
import numpy as np
A = np.arange(1, 7).reshape(2, 3)
B = np.arange(1, 7).reshape(3, 2)
print("A @ B =\n", A @ B)
print("A * A =\n",A * A)

X = np.array([[1., 200., 3.], [2., 180., 5.],
             [3., 220., 4.], [4., 200., 8.]])
Z= (X-X.mean(axis=0))/ X.std(axis=0)
print(Z.mean(axis=0).round(6), Z.std(axis=0).round(6))
u = np.array([1., 2., 2.]); q = np.array([2., 0., 1.])
cos = np.dot(u, q)/ (np.linalg.norm(u) * np.linalg.norm(q))
print(round(float(cos),4))

A @ B =
 [[22 28]
 [49 64]]
A * A =
 [[ 1  4  9]
 [16 25 36]]
[0. 0. 0.] [1. 1. 1.]
0.5963


In [13]:
G = {"Entry": {"Hall": 2, "Stairs": 5},
     "Hall": {"Lab": 4, "Office": 1},
     "Stairs": {"ServerRoom": 3},
     "Office": {"Lab": 1, "ServerRoom": 7},
     "Lab": {"ServerRoom":2 },
     "ServerRoom":{}}

In [21]:
from collections import deque
import heapq
def bfs(start, goal):
    frontier, parent, expanded = deque([start]), {start: None}, 0
    while frontier:
        s = frontier.popleft(); expanded += 1
        if s ==goal: break
        for n in G[s]:
            if n not in parent:
                parent[n] = s
                frontier.append(n)
    if goal not in parent:
        return None, None, expanded
    path, s = [], goal
    while s is not None:
        path.append(s)
        s = parent[s]
    path.reverse()
    cost = sum(G[a][b] for a, b in zip (path, path[1:]))
    return path, cost, expanded
def ucs(start, goal):
    pq, done, expanded = [(0, start, [start])], set(), 0
    while pq:
        g, s, path = heapq.heappop(pq)
        if s in done: continue
        done.add(s); expanded += 1
        if s == goal: return path, g, expanded
        for n, c in G[s].items():
            if n not in done:
                heapq.heappush(pq, (g + c, n, path + [n]))
    return None, None, expanded
for name, algo in [("BFS", bfs), ("UCS", ucs)]:
    path, cost, expanded = algo("Entry", "ServerRoom")
    print(f"{name} path:", " -> ".join(path))
    print(f"{name} total cost:", cost)
    print(f"{name} nodes expanded:", expanded)
    print()

BFS path: Entry -> Stairs -> ServerRoom
BFS total cost: 8
BFS nodes expanded: 6

UCS path: Entry -> Hall -> Office -> Lab -> ServerRoom
UCS total cost: 6
UCS nodes expanded: 6

